# Socratic Math Tutor: Kaggle runner (run as **Save Version → Save & Run All (Commit)**)

Run this notebook **3 times** (3 committed versions). Each part stays well under Kaggle's 12 h limit, and each part resumes
from the previous part's saved output:

| Part | `STAGES` | `RESUME_DIR` | ≈ GPU time |
|---|---|---|---|
| 1 | `data train ablations` | `""` | 5–6 h |
| 2 | `dpo eval judge` | output of Part 1 | 3–4 h |
| 3 | `sim gsm8k report` | output of Part 2 | 2–3 h |

Settings (right panel): Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = Files (optional).

In [ ]:
# ======== EDIT THESE TWO LINES FOR EACH PART ========
STAGES = "data train ablations"      # Part 2: "dpo eval judge"   Part 3: "sim gsm8k report"
RESUME_DIR = ""                      # Part 2/3: "auto" (searches all attached inputs) or an explicit path
REPO_URL = "https://github.com/<YOUR_USERNAME>/socratic-tutor-app.git"
# =====================================================

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("socratic-tutor-app"):
    if os.path.exists("/kaggle/input/socratic-tutor-app-code"):          # option B: code uploaded as a Kaggle Dataset
        subprocess.run("cp -r /kaggle/input/socratic-tutor-app-code /kaggle/working/socratic-tutor-app", shell=True, check=True)
    else:                                                                 # option A: public GitHub repo
        subprocess.run(f"git clone {REPO_URL}", shell=True, check=True)
os.chdir("/kaggle/working/socratic-tutor-app")
# Optional: Add-ons -> Secrets -> HF_TOKEN (faster, rate-limit-free model downloads)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print(os.getcwd())

In [ ]:
# Resume: restore data/ + outputs/ from the previous part (attached under /kaggle/input)
import shutil, subprocess, zipfile

def find(name, parent=None):
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if name in files and (parent is None or os.path.basename(root) == parent):
            return os.path.join(root, name)
    return None

if RESUME_DIR:
    hit = None if RESUME_DIR == "auto" else (f"{RESUME_DIR}/data/sft_socratic_train.jsonl"
                                              if os.path.exists(f"{RESUME_DIR}/data/sft_socratic_train.jsonl") else None)
    hit = hit or find("sft_socratic_train.jsonl", parent="data")
    if hit:                                   # full saved folders (saved/ from the last cell)
        src = os.path.dirname(os.path.dirname(hit))
        print("Resuming from folders:", src)
        for d in ("data", "outputs"):
            if os.path.exists(f"{src}/{d}"):
                shutil.copytree(f"{src}/{d}", d, dirs_exist_ok=True)
    else:                                     # fallback: the results zip + regenerate data deterministically
        z = find("socratic_results.zip")
        assert z, "No previous output found under /kaggle/input - attach the notebook version that ran Part 1."
        print("Resuming from zip:", z)
        zipfile.ZipFile(z).extractall(".")
        subprocess.run("bash scripts/run_pipeline.sh data", shell=True, check=True)   # same seed -> identical splits
    assert os.path.exists("data/sft_socratic_train.jsonl"), "data/ was not restored"
    assert os.path.exists("outputs/sft_socratic/adapter"), "outputs/sft_socratic/adapter missing: Part 1 training did not finish"
!ls data; ls outputs 2>/dev/null; true

In [ ]:
!pip install -q unsloth
!pip install -q -r requirements-train.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m pytest -q tests   # sanity checks (CPU, a few seconds)

In [ ]:
!bash scripts/run_pipeline.sh {STAGES} 2>&1 | tee -a pipeline.log

In [ ]:
# Show what was produced in this part
!cat outputs/*/train_metrics.json 2>/dev/null | grep -E '"run"|n_train|train_hours|peak_vram|final_eval_loss' ; true
!tail -5 outputs_sft_raw.log outputs_sft_full_loss.log 2>/dev/null; true
if os.path.exists("results/RESULTS.md"):
    from IPython.display import Markdown, display
    display(Markdown(open("results/RESULTS.md").read()))
    display(Markdown(open("results/resume_bullets.md").read()))

In [ ]:
# Save data/ + outputs/ OUTSIDE the git repo folder so Kaggle keeps them in the version output
!rm -rf outputs/*/checkpoints
!mkdir -p /kaggle/working/saved && cp -r data outputs /kaggle/working/saved/ 2>/dev/null; true
# One small zip for downloading to your Mac (adapters + metrics + eval outputs + results, no merged 3 GB model)
!zip -qr /kaggle/working/socratic_results.zip results outputs/*/adapter outputs/*/train_metrics.json outputs/eval data/data_stats.json data/eval_items.jsonl 2>/dev/null; true
!du -sh /kaggle/working/saved/* /kaggle/working/*.zip 2>/dev/null